# Dimensionality Reduction - Interactive Development

Explore metabolite patterns using dimensionality reduction techniques.

**Workflow:**
1. Run Setup (Section 1)
2. Load Data (Section 2)
3. Choose Algorithm & Adjust Parameters (Section 3)
4. Generate Plot (Section 4) - **Re-run to see parameter changes**
5. Export when satisfied (Section 5)

**Supported Algorithms:**
- **PCA**: Linear, fast, interpretable (variance explained)
- **t-SNE**: Non-linear, good for visualization
- **UMAP**: Non-linear, faster than t-SNE, preserves global structure

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our dimensionality reduction module
from dimensionality_reduction import (
    DimensionalityReducer,
    load_integrated_results_for_dr,
    quick_pca
)

# High-resolution plots
%config InlineBackend.figure_format = 'retina'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

print("✓ Setup complete")

## 2. Load Data

In [ ]:
# Specify data path
DATA_PATH = '../output/4_groups/integrated_results_cortex.csv'

# Load data
data, sample_groups = load_integrated_results_for_dr(DATA_PATH)

print(f"✓ Loaded data from {Path(DATA_PATH).name}")
print(f"\nData shape: {data.shape[0]} samples × {data.shape[1]} features (m/z bins)")

# Show sample groups
print("\nSample groups:")
for group in sorted(set(sample_groups.values())):
    samples = [s for s, g in sample_groups.items() if g == group]
    print(f"  {group}: {len(samples)} samples ({', '.join(samples)})")

print("\nFirst few features:")
display(data.iloc[:, :5])

## 3. Configure Parameters

**Adjust these parameters and re-run Section 4 to see changes**

In [ ]:
# ============================================================
# ALGORITHM SELECTION
# ============================================================

ALGORITHM = 'PCA'  # 'PCA', 't-SNE', or 'UMAP'

# ============================================================
# DATA PREPROCESSING
# ============================================================

# Normalization method
NORMALIZATION = 'zscore'  # 'zscore', 'minmax', 'robust', 'none'
                          # zscore: mean=0, std=1 (recommended)
                          # minmax: scale to 0-1 range
                          # robust: use median and IQR (robust to outliers)
                          # none: use raw values

# Log transformation (apply before normalization)
LOG_TRANSFORM = False     # True: apply log2(x+1) transformation

# ============================================================
# PCA PARAMETERS
# ============================================================

PCA_N_COMPONENTS = 2      # Number of principal components (2 for visualization)
PCA_RANDOM_STATE = 42     # Random seed for reproducibility

# ============================================================
# t-SNE PARAMETERS
# ============================================================

TSNE_N_COMPONENTS = 2     # Number of dimensions (usually 2)
TSNE_PERPLEXITY = 30.0    # Perplexity (5-50 recommended)
                          # Lower: focus on local structure
                          # Higher: focus on global structure
TSNE_LEARNING_RATE = 'auto'  # Learning rate ('auto' or 10-1000)
TSNE_N_ITER = 1000        # Number of iterations (250-1000 recommended)
TSNE_RANDOM_STATE = 42    # Random seed

# ============================================================
# UMAP PARAMETERS
# ============================================================

UMAP_N_COMPONENTS = 2     # Number of dimensions (usually 2)
UMAP_N_NEIGHBORS = 15     # Number of neighbors (2-100, default 15)
                          # Lower: focus on local structure
                          # Higher: focus on global structure
UMAP_MIN_DIST = 0.1       # Minimum distance (0.0-0.99, default 0.1)
                          # Lower: tighter clusters
                          # Higher: looser clusters
UMAP_METRIC = 'euclidean' # Distance metric: 'euclidean', 'manhattan', 'cosine'
UMAP_RANDOM_STATE = 42    # Random seed

# ============================================================
# VISUALIZATION SETTINGS
# ============================================================

# Figure size
FIG_WIDTH = 10
FIG_HEIGHT = 8

# Point appearance
POINT_SIZE = 100          # Size of scatter points
POINT_ALPHA = 0.7         # Transparency (0=transparent, 1=opaque)

# Labels
SHOW_LABELS = True        # Show sample labels on plot
LABEL_FONTSIZE = 9        # Font size for labels

# Colors
COLOR_PALETTE = 'Set2'    # Color palette:
                          # 'Set2', 'Set1', 'Paired', 'tab10', 'husl'

# Legend
LEGEND_LOC = 'best'       # Legend location: 'best', 'upper right', etc.

# Title
TITLE = None              # Plot title (None = auto-generate)

# Variance explained (PCA only)
SHOW_VARIANCE = True      # Show variance explained in axis labels

print("✓ Parameters configured")
print(f"  Algorithm: {ALGORITHM}")
print(f"  Normalization: {NORMALIZATION}")
print(f"  Log transform: {LOG_TRANSFORM}")

## 4. Generate Dimensionality Reduction Plot

**Re-run this cell after changing parameters in Section 3**

In [ ]:
# Create DimensionalityReducer instance
reducer = DimensionalityReducer(data, sample_groups)

# Normalize data
reducer.normalize_data(method=NORMALIZATION, log_transform=LOG_TRANSFORM)

# Apply selected algorithm
if ALGORITHM == 'PCA':
    reducer.apply_pca(
        n_components=PCA_N_COMPONENTS,
        random_state=PCA_RANDOM_STATE
    )
    print(f"✓ Applied PCA")
    
    # Show variance explained
    if hasattr(reducer.model, 'explained_variance_ratio_'):
        var_ratio = reducer.model.explained_variance_ratio_
        print(f"\nVariance explained:")
        for i, var in enumerate(var_ratio[:PCA_N_COMPONENTS]):
            print(f"  PC{i+1}: {var*100:.2f}%")
        print(f"  Total: {sum(var_ratio[:PCA_N_COMPONENTS])*100:.2f}%")

elif ALGORITHM == 't-SNE':
    reducer.apply_tsne(
        n_components=TSNE_N_COMPONENTS,
        perplexity=TSNE_PERPLEXITY,
        learning_rate=TSNE_LEARNING_RATE,
        n_iter=TSNE_N_ITER,
        random_state=TSNE_RANDOM_STATE
    )
    print(f"✓ Applied t-SNE")
    print(f"  Perplexity: {TSNE_PERPLEXITY}")
    print(f"  Iterations: {TSNE_N_ITER}")

elif ALGORITHM == 'UMAP':
    try:
        reducer.apply_umap(
            n_components=UMAP_N_COMPONENTS,
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            metric=UMAP_METRIC,
            random_state=UMAP_RANDOM_STATE
        )
        print(f"✓ Applied UMAP")
        print(f"  Neighbors: {UMAP_N_NEIGHBORS}")
        print(f"  Min distance: {UMAP_MIN_DIST}")
    except ImportError as e:
        print("❌ UMAP not installed. Install with: pip install umap-learn")
        raise

else:
    raise ValueError(f"Unknown algorithm: {ALGORITHM}")

# Generate plot
fig = reducer.plot_2d(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    point_size=POINT_SIZE,
    alpha=POINT_ALPHA,
    show_labels=SHOW_LABELS,
    label_fontsize=LABEL_FONTSIZE,
    title=TITLE,
    legend_loc=LEGEND_LOC,
    palette=COLOR_PALETTE,
    show_variance=SHOW_VARIANCE
)

plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {data.shape[0]}")
print(f"Features: {data.shape[1]}")
print(f"Groups: {len(set(sample_groups.values()))}")
print(f"Normalization: {NORMALIZATION}")
print(f"Log transform: {LOG_TRANSFORM}")
print(f"{'='*60}")

## 5. Variance Explained Plot (PCA only)

Visualize how much variance is explained by each principal component.

In [ ]:
if ALGORITHM == 'PCA':
    # Show variance explained for more components
    N_COMPONENTS_TO_SHOW = min(10, data.shape[1])  # Show up to 10 components
    
    # Re-run PCA with more components for variance analysis
    reducer_var = DimensionalityReducer(data, sample_groups)
    reducer_var.normalize_data(method=NORMALIZATION, log_transform=LOG_TRANSFORM)
    reducer_var.apply_pca(n_components=N_COMPONENTS_TO_SHOW)
    
    # Plot variance explained
    fig_var = reducer_var.plot_variance_explained(
        n_components=N_COMPONENTS_TO_SHOW,
        figsize=(12, 5)
    )
    plt.show()
    
    # Print cumulative variance
    var_ratio = reducer_var.model.explained_variance_ratio_
    cumsum = np.cumsum(var_ratio)
    print("\nCumulative variance explained:")
    for i in range(min(5, len(cumsum))):
        print(f"  PC1-PC{i+1}: {cumsum[i]*100:.2f}%")
else:
    print(f"Variance explained plot is only available for PCA (current: {ALGORITHM})")

## 6. Export Figure and Data

Run this cell when you're satisfied with the plot to save it.

In [ ]:
# Output settings
OUTPUT_DIR = Path('../output/4_groups/dimensionality_reduction')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# File naming
roi_name = Path(DATA_PATH).stem.replace('integrated_results_', '')
base_name = f'{ALGORITHM.lower()}_{roi_name}'

# Save figure as PNG (high resolution)
png_path = OUTPUT_DIR / f'{base_name}.png'
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PNG: {png_path}")

# Save figure as PDF (vector, for publications)
pdf_path = OUTPUT_DIR / f'{base_name}.pdf'
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PDF: {pdf_path}")

# Save variance plot if PCA
if ALGORITHM == 'PCA':
    var_png_path = OUTPUT_DIR / f'{base_name}_variance.png'
    fig_var.savefig(var_png_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved variance plot: {var_png_path}")

# Export data
data_export_path = OUTPUT_DIR / f'{base_name}_data'
reducer.export_data(
    str(data_export_path),
    include_original=True,
    include_normalized=True,
    include_reduced=True
)

print(f"\n✓ All files saved to: {OUTPUT_DIR}")

## 7. Quick Reference: Parameter Guidelines

### Algorithm Selection

**PCA (Principal Component Analysis)**
- ✅ Fast, deterministic, interpretable
- ✅ Shows variance explained
- ✅ Good for initial exploration
- ❌ Linear method (may miss non-linear patterns)
- **Use when**: You want interpretable results and variance information

**t-SNE (t-Distributed Stochastic Neighbor Embedding)**
- ✅ Excellent for visualization
- ✅ Reveals non-linear patterns
- ❌ Slow for large datasets
- ❌ Non-deterministic (different runs give different results)
- ❌ Distances between clusters not meaningful
- **Use when**: You want to visualize complex patterns

**UMAP (Uniform Manifold Approximation and Projection)**
- ✅ Fast (faster than t-SNE)
- ✅ Preserves both local and global structure
- ✅ More deterministic than t-SNE
- ❌ Requires installation: `pip install umap-learn`
- **Use when**: You want t-SNE-like results but faster

---

### Parameter Tuning

#### PCA Parameters
```python
PCA_N_COMPONENTS = 2  # Usually 2 for visualization
```

#### t-SNE Parameters
```python
# Perplexity: balance between local and global structure
TSNE_PERPLEXITY = 5    # Focus on local structure (small datasets)
TSNE_PERPLEXITY = 30   # Balanced (default, recommended)
TSNE_PERPLEXITY = 50   # Focus on global structure (large datasets)

# Iterations: more = better convergence
TSNE_N_ITER = 250   # Quick test
TSNE_N_ITER = 1000  # Recommended
TSNE_N_ITER = 5000  # High quality (slow)

# Learning rate
TSNE_LEARNING_RATE = 'auto'  # Recommended
TSNE_LEARNING_RATE = 200     # Manual (10-1000)
```

#### UMAP Parameters
```python
# n_neighbors: balance between local and global structure
UMAP_N_NEIGHBORS = 5    # Focus on local structure
UMAP_N_NEIGHBORS = 15   # Balanced (default)
UMAP_N_NEIGHBORS = 50   # Focus on global structure

# min_dist: cluster tightness
UMAP_MIN_DIST = 0.0   # Very tight clusters
UMAP_MIN_DIST = 0.1   # Balanced (default)
UMAP_MIN_DIST = 0.5   # Loose clusters
```

---

### Normalization Methods
```python
NORMALIZATION = 'zscore'  # Mean=0, std=1 (recommended)
NORMALIZATION = 'minmax'  # Scale to 0-1
NORMALIZATION = 'robust'  # Median and IQR (robust to outliers)
NORMALIZATION = 'none'    # No normalization
```

---

### Troubleshooting

**Problem: All points clustered together**
- Try different normalization method
- For t-SNE: decrease perplexity
- For UMAP: decrease n_neighbors

**Problem: Points too spread out**
- For t-SNE: increase perplexity
- For UMAP: increase n_neighbors or decrease min_dist

**Problem: t-SNE results change every time**
- This is normal! Set TSNE_RANDOM_STATE to a fixed number for reproducibility
- Run multiple times and look for consistent patterns

**Problem: UMAP not installed**
```bash
pip install umap-learn
```

## 8. Compare Multiple Algorithms

Generate plots for all three algorithms side-by-side for comparison.

In [ ]:
# Create figure with subplots
fig_compare, axes = plt.subplots(1, 3, figsize=(18, 5))

algorithms = ['PCA', 't-SNE', 'UMAP']
reducers = []

for i, algo in enumerate(algorithms):
    print(f"\nGenerating {algo}...")
    
    # Create reducer
    r = DimensionalityReducer(data, sample_groups)
    r.normalize_data(method=NORMALIZATION, log_transform=LOG_TRANSFORM)
    
    # Apply algorithm
    try:
        if algo == 'PCA':
            r.apply_pca(n_components=2, random_state=42)
        elif algo == 't-SNE':
            r.apply_tsne(n_components=2, perplexity=30, n_iter=1000, random_state=42)
        elif algo == 'UMAP':
            r.apply_umap(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
        
        reducers.append(r)
        
        # Plot on subplot
        ax = axes[i]
        
        # Get unique groups and colors
        unique_groups = sorted(set(sample_groups.values()))
        colors = sns.color_palette(COLOR_PALETTE, n_colors=len(unique_groups))
        group_colors = dict(zip(unique_groups, colors))
        
        # Plot each group
        for group in unique_groups:
            group_samples = [s for s, g in sample_groups.items() if g == group]
            group_data = r.reduced_data.loc[group_samples]
            
            ax.scatter(
                group_data.iloc[:, 0],
                group_data.iloc[:, 1],
                c=[group_colors[group]],
                s=POINT_SIZE,
                alpha=POINT_ALPHA,
                label=group,
                edgecolors='black',
                linewidths=0.5
            )
        
        # Set labels
        ax.set_xlabel(r.reduced_data.columns[0], fontweight='bold')
        ax.set_ylabel(r.reduced_data.columns[1], fontweight='bold')
        ax.set_title(algo, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle=':')
        
        if i == 2:  # Add legend to last plot
            ax.legend(loc='best', frameon=True, fancybox=True, shadow=True)
        
        print(f"  ✓ {algo} complete")
        
    except ImportError:
        print(f"  ❌ {algo} not available (install umap-learn)")
        axes[i].text(0.5, 0.5, f'{algo}\nNot Available', 
                    ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_xticks([])
        axes[i].set_yticks([])

plt.tight_layout()
plt.show()

# Save comparison plot
comparison_path = OUTPUT_DIR / f'comparison_{roi_name}.png'
fig_compare.savefig(comparison_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Saved comparison plot: {comparison_path}")